In [49]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from scipy.optimize import linprog
import os

In [50]:
CONFIG = {

    # macro coverage targets for sequential method (fallback)
    "fiber_coverage_ratio": 0.50,
    "protein_coverage_ratio": 0.55,

    # tolerance: 85-115% of target is acceptable
    "macro_tolerance_low": 0.85,
    "macro_tolerance_high": 1.15,

    # quantity bounds (grams)
    "qty_min": 30,
    "qty_max": 250,
    "side_qty_min": 50,
    "side_qty_max": 80,

    # history / repetition
    "repetition_penalty_weight": 0.30,
    "recent_days_limit": 3,
    "unique_food_reset_ratio": 0.50,
    "history_reset_days": 7,

    # health thresholds
    "high_hba1c": 6.5,
    "high_triglycerides": 150,
    "high_ldl": 130,
    "high_bp_sys": 140,
    "high_bp_dia": 90,
    "gi_low": 55,

    # KNN: compare user meal targets to each food's per-100g row by MACRO COMPOSITION
    # (density): L1-normalize [carb, protein, fiber, fat] then weighted cosine similarity.
    # No reference portion / quantity — only "shape" of the macro profile must align.
    "knn_neighbors": 5,  # kept for optional top-k inspection; ranking uses all candidates
    "knn_composition_eps": 1e-9,
    "knn_slot_weights": {
        "fiber": [0.9, 0.9, 2.0, 0.9],
        "protein": [0.9, 2.0, 0.9, 0.9],
        "carb": [2.0, 0.9, 0.9, 0.9],
        "snack": [1.0, 1.0, 1.0, 1.0],
    },

    # file
    "history_file": "recommendation_history.csv"
}

In [51]:
class RecommendationHistory:

    def __init__(self):
        self.file = CONFIG["history_file"]
        if os.path.exists(self.file) and os.path.getsize(self.file) > 0:
            try:
                self.history = pd.read_csv(self.file)
            except pd.errors.EmptyDataError:
                self.history = pd.DataFrame(columns=["user_id", "food_id", "date", "feedback"])
        else:
            self.history = pd.DataFrame(columns=["user_id", "food_id", "date", "feedback"])
            self.history.to_csv(self.file, index=False)

    def save(self):
        self.history.to_csv(self.file, index=False)

    def reset_if_needed(self, user_id, total_foods):
        user_history = self.history[self.history["user_id"] == user_id]
        if len(user_history) == 0:
            return

        user_history = user_history.copy()
        user_history["date"] = pd.to_datetime(user_history["date"])
        last_date = user_history["date"].max()

        if (datetime.now() - last_date).days >= CONFIG["history_reset_days"]:
            self.history = self.history[self.history["user_id"] != user_id]
            self.save()
            return

        unique_foods = user_history["food_id"].nunique()
        if unique_foods > total_foods * CONFIG["unique_food_reset_ratio"]:
            self.history = self.history[self.history["user_id"] != user_id]
            self.save()

    def add(self, user_id, food_id, feedback="shown"):
        new_row = pd.DataFrame([{
            "user_id": user_id,
            "food_id": food_id,
            "date": datetime.now(),
            "feedback": feedback
        }])
        self.history = pd.concat([self.history, new_row], ignore_index=True)
        self.save()

    def get_penalty(self, user_id, food_id):
        df = self.history[
            (self.history["user_id"] == user_id) &
            (self.history["food_id"] == food_id)
        ]
        if df.empty:
            return 0.0

        df = df.copy()
        df["date"] = pd.to_datetime(df["date"])

        penalty = len(df) * CONFIG["repetition_penalty_weight"]

        last_date = df["date"].max()
        if (datetime.now() - last_date).days <= CONFIG["recent_days_limit"]:
            penalty += 0.5

        dislike_penalty = len(df[df["feedback"] == "dislike"]) * 2.0
        like_bonus = len(df[df["feedback"] == "like"]) * -0.3

        return penalty + dislike_penalty + like_bonus

In [52]:
class FoodRecommender:

    def __init__(self, food_df):
        self.food_df = food_df
        self.history = RecommendationHistory()

    # =====================================================
    # HEALTH FILTERING
    # =====================================================

    def filter_foods(self, user):
        df = self.food_df.copy()

        if user["veg_only"]:
            df = df[df["is_veg"] == 1]

        meal = user["meal_type"]
        # handle "snacks" -> column is "is_snack"
        col = f"is_{meal}" if f"is_{meal}" in df.columns else f"is_{meal.rstrip('s')}"
        if col in df.columns:
            df = df[df[col] == 1]

        if user["hba1c_percent"] >= CONFIG["high_hba1c"]:
            df = df[df["glycemic_index"] <= CONFIG["gi_low"]]

        if user["triglycerides_mg_dl"] >= CONFIG["high_triglycerides"]:
            df = df[df["is_high_saturated_fat"] == 0]

        if user["ldl_cholesterol_mg_dl"] >= CONFIG["high_ldl"]:
            df = df[df["is_high_saturated_fat"] == 0]

        if user["systolic_bp_mmHg"] >= CONFIG["high_bp_sys"] or \
           user["diastolic_bp_mmHg"] >= CONFIG["high_bp_dia"]:
            df = df[df["is_high_sodium"] == 0]

        return df

    # =====================================================
    # Macro composition match (density) — no quantity / portion scaling
    # =====================================================

    def _macro_vector(self, carb, protein, fiber, fat):
        v = np.array([carb, protein, fiber, fat], dtype=float)
        return np.maximum(v, 0.0)

    def _composition(self, v):
        """L1-normalize: compare meal targets (g) with per-100g rows on the same simplex."""
        eps = float(CONFIG.get("knn_composition_eps", 1e-9))
        s = float(v.sum())
        if s <= eps:
            return np.ones(4, dtype=float) / 4.0
        return v / s

    def _slot_weights(self, macro_type):
        w = CONFIG["knn_slot_weights"].get(macro_type)
        if w is None:
            w = CONFIG["knn_slot_weights"]["snack"]
        return np.array(w, dtype=float)

    def _weighted_cosine_batch(self, u_comp, F, weights):
        """
        Weighted cosine similarity between u_comp (length 4) and each row of F (n,4).
        Higher = better profile match.
        """
        w = np.maximum(np.asarray(weights, dtype=float), 1e-9)
        su = np.sqrt(w) * u_comp
        den_u = np.linalg.norm(su)
        if den_u <= 1e-12:
            return np.zeros(F.shape[0])
        SF = np.sqrt(w) * F
        num = SF @ su
        den_f = np.linalg.norm(SF, axis=1)
        den = den_u * np.maximum(den_f, 1e-12)
        return num / den

    def _targets_to_user_vec(self, targets):
        if isinstance(targets, dict):
            return self._macro_vector(
                targets["carb"], targets["protein"],
                targets["fiber"], targets["fat"],
            )
        return np.asarray(targets, dtype=float).reshape(-1)

    # =====================================================
    # KNN-BASED FOOD SELECTION (composition similarity)
    # =====================================================

    def select_food_knn(self, df, macro_type, user_id, targets=None, ideal_vector=None):
        """
        Pick foods whose macro density (fraction of carb/protein/fiber/fat mass) best matches
        the user's meal targets. Per-100g food rows and meal targets both map to the same
        composition space — no scaling by portion size.

        Pass `targets` (dict, g per meal) or `ideal_vector` ([carb, protein, fiber, fat] per 100g).
        """
        foods = df[df["macro_type"] == macro_type].copy()

        if len(foods) == 0:
            return None

        if targets is not None:
            user_raw = self._targets_to_user_vec(targets)
        elif ideal_vector is not None:
            user_raw = np.asarray(ideal_vector, dtype=float).reshape(-1)
        else:
            raise ValueError("select_food_knn requires either targets or ideal_vector")

        if user_raw.shape[0] != 4:
            raise ValueError("Macro vector must have length 4: carb, protein, fiber, fat")

        F = foods[["carb_g", "protein_g", "fiber_g", "fat_g"]].values.astype(float)
        row_sums = F.sum(axis=1)
        eps = CONFIG.get("knn_composition_eps", 1e-9)
        valid = row_sums > eps
        if not np.any(valid):
            return None

        foods = foods.loc[valid].copy()
        F = F[valid]

        comp_food = F / np.maximum(row_sums[valid, np.newaxis], 1e-12)
        u_comp = self._composition(user_raw)
        weights = self._slot_weights(macro_type)
        sims = self._weighted_cosine_batch(u_comp, comp_food, weights)

        foods["_match"] = sims
        foods["penalty"] = foods["food_id"].apply(
            lambda fid: self.history.get_penalty(user_id, fid)
        )
        foods["score"] = (1.0 - foods["_match"]) + foods["penalty"] * 5.0
        foods = foods.sort_values("score", ascending=True)

        return foods.iloc[0]


    # =====================================================
    # MACRO VECTOR PER GRAM
    # =====================================================

    def macro_per_gram(self, food):
        return np.array([
            food["carb_g"] / 100.0,
            food["protein_g"] / 100.0,
            food["fiber_g"] / 100.0,
            food["fat_g"] / 100.0
        ])

    # =====================================================
    # LINEAR PROGRAMMING QUANTITY OPTIMIZER
    # =====================================================

    def solve_quantities_lp(self, foods, targets):
        """
        Minimize total macro error using Linear Programming.

        Decision variables: qty_1 ... qty_n (grams for each food)

        We minimize a slack variable that represents deviation.
        Since scipy linprog only does LP (not quadratic), we use
        an auxiliary formulation:

        For each macro m:
            sum(food_i_macro_m_per_gram * qty_i) + slack_m_pos - slack_m_neg = target_m

        Minimize: sum(slack_pos + slack_neg)

        Bounds: qty_min <= qty_i <= qty_max
                0 <= slack >= 0
        """
        n = len(foods)
        m = 4  # carb, protein, fiber, fat

        target_vec = np.array([
            targets["carb"],
            targets["protein"],
            targets["fiber"],
            targets["fat"]
        ], dtype=float)

        # build macro matrix: A[macro][food] = macro_per_gram
        A_macro = np.zeros((m, n))
        for j, food in enumerate(foods):
            A_macro[:, j] = self.macro_per_gram(food)

        # decision vars: [qty_0..qty_n-1, slack_pos_0..3, slack_neg_0..3]
        # total vars = n + 2*m
        num_vars = n + 2 * m

        # objective: minimize sum of slack vars (weighted)
        # weight macros by importance: carb=1.5, protein=1.5, fiber=1.0, fat=1.0
        weights = np.array([1.5, 1.5, 1.0, 1.0])
        c = np.zeros(num_vars)
        for i in range(m):
            c[n + i] = weights[i]        # slack_pos
            c[n + m + i] = weights[i]    # slack_neg

        # equality constraints: A_macro @ qty + slack_pos - slack_neg = target
        A_eq = np.zeros((m, num_vars))
        b_eq = np.zeros(m)

        for i in range(m):
            for j in range(n):
                A_eq[i, j] = A_macro[i, j]
            A_eq[i, n + i] = 1.0       # slack_pos
            A_eq[i, n + m + i] = -1.0  # slack_neg
            b_eq[i] = target_vec[i]

        # bounds
        bounds = []
        for j in range(n):
            bounds.append((CONFIG["qty_min"], CONFIG["qty_max"]))
        for i in range(2 * m):
            bounds.append((0, None))  # slacks >= 0

        try:
            result = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')

            if result.success:
                quantities = result.x[:n]
                return np.clip(quantities, CONFIG["qty_min"], CONFIG["qty_max"])
        except Exception:
            pass

        # fallback to lstsq
        return self.solve_quantities_fallback(foods, targets)

    # =====================================================
    # FALLBACK: LEAST SQUARES SOLVER
    # =====================================================

    def solve_quantities_fallback(self, foods, targets):
        A = np.array([self.macro_per_gram(f) for f in foods]).T

        b = np.array([
            targets["carb"],
            targets["protein"],
            targets["fiber"],
            targets["fat"]
        ])

        try:
            qty = np.linalg.lstsq(A, b, rcond=None)[0]
        except Exception:
            qty = np.ones(len(foods)) * 100.0

        return np.clip(qty, CONFIG["qty_min"], CONFIG["qty_max"])

    # =====================================================
    # SEQUENTIAL QUANTITY ESTIMATION (SAFETY CAPPED)
    # =====================================================

    def solve_quantities_sequential(self, foods, targets):
        """
        Step-by-step quantity computation with macro subtraction
        and overflow protection. Used as a secondary fallback
        and to validate LP results.
        """
        remaining = {
            "carb": targets["carb"],
            "protein": targets["protein"],
            "fiber": targets["fiber"],
            "fat": targets["fat"]
        }

        quantities = []
        macro_keys = ["carb", "protein", "fiber", "fat"]
        primary_macro = {
            "fiber": "fiber",
            "protein": "protein",
            "carb": "carb"
        }

        for food in foods:
            mtype = food.get("macro_type", "carb")
            pmacro = primary_macro.get(mtype, "carb")

            macro_col = f"{pmacro}_g"
            per100 = food[macro_col]

            if per100 <= 0:
                quantities.append(CONFIG["qty_min"])
                continue

            # compute quantity from primary macro
            target_val = remaining[pmacro]
            if mtype in ["fiber", "protein"]:
                target_val *= CONFIG.get(f"{pmacro}_coverage_ratio", 0.55)

            qty = (target_val / per100) * 100.0

            # check all macros don't overflow
            per_gram = self.macro_per_gram(food)
            for idx, key in enumerate(macro_keys):
                if per_gram[idx] > 0 and remaining[key] > 0:
                    max_qty = remaining[key] / per_gram[idx]
                    qty = min(qty, max_qty)

            qty = np.clip(qty, CONFIG["qty_min"], CONFIG["qty_max"])
            quantities.append(qty)

            # subtract contributed macros
            for idx, key in enumerate(macro_keys):
                remaining[key] -= per_gram[idx] * qty
                remaining[key] = max(remaining[key], 0)

        return np.array(quantities)

    # =====================================================
    # VALIDATE QUANTITIES (overflow check)
    # =====================================================

    def validate_and_fix(self, foods, quantities, targets):
        """
        After computing quantities, verify no macro exceeds 115% of target.
        If it does, scale down proportionally.
        """
        macro_keys = ["carb", "protein", "fiber", "fat"]
        target_vec = np.array([
            targets["carb"], targets["protein"],
            targets["fiber"], targets["fat"]
        ])

        # compute actual macros
        actual = np.zeros(4)
        for i, food in enumerate(foods):
            actual += self.macro_per_gram(food) * quantities[i]

        # check overflow
        for idx in range(4):
            if target_vec[idx] > 0:
                ratio = actual[idx] / target_vec[idx]
                if ratio > CONFIG["macro_tolerance_high"]:
                    # scale down ALL quantities proportionally
                    scale = CONFIG["macro_tolerance_high"] / ratio
                    quantities = quantities * scale

        quantities = np.clip(quantities, CONFIG["qty_min"], CONFIG["qty_max"])
        return quantities

    # =====================================================
    # SNACK RECOMMENDATION
    # =====================================================

    def recommend_snack(self, df, user_id, targets):
        user_raw = self._targets_to_user_vec(targets)
        target_vec = np.array([
            targets["carb"], targets["protein"],
            targets["fiber"], targets["fat"]
        ])

        F = df[["carb_g", "protein_g", "fiber_g", "fat_g"]].values.astype(float)
        row_sums = F.sum(axis=1)
        eps = CONFIG.get("knn_composition_eps", 1e-9)
        valid = row_sums > eps
        if not np.any(valid):
            return []

        d = df.loc[valid].copy()
        F = F[valid]
        comp_food = F / np.maximum(row_sums[valid, np.newaxis], 1e-12)
        u_comp = self._composition(user_raw)
        weights = self._slot_weights("snack")
        sims = self._weighted_cosine_batch(u_comp, comp_food, weights)

        d["_match"] = sims
        d["penalty"] = d["food_id"].apply(
            lambda fid: self.history.get_penalty(user_id, fid)
        )
        d["score"] = (1.0 - d["_match"]) + d["penalty"] * 5.0
        d = d.sort_values("score", ascending=True)

        best_food = d.iloc[0]

        macro = self.macro_per_gram(best_food)
        denom = np.dot(macro, macro)

        if denom > 0:
            qty = np.dot(target_vec, macro) / denom
        else:
            qty = 100.0

        for idx, key in enumerate(["carb", "protein", "fiber", "fat"]):
            if macro[idx] > 0 and targets[key] > 0:
                max_qty = (targets[key] * CONFIG["macro_tolerance_high"]) / macro[idx]
                qty = min(qty, max_qty)

        qty = np.clip(qty, CONFIG["qty_min"], CONFIG["qty_max"])

        return [(
            best_food["food_name"],
            round(float(qty), 1),
            best_food["serving_unit"],
            best_food["food_id"]
        )]


    # =====================================================
    # MAIN RECOMMEND
    # =====================================================

    def recommend(self, user_id, user):
        df = self.filter_foods(user)

        if len(df) == 0:
            print("No foods available after health filtering.")
            return []

        self.history.reset_if_needed(user_id, len(self.food_df))

        targets = {
            "carb": user["target_carbs_g"],
            "protein": user["target_protein_g"],
            "fiber": user["target_fiber_g"],
            "fat": user["target_fat_g"]
        }

        # ------ SNACK ------
        if user["meal_type"] in ["snack", "snacks"]:
            return self.recommend_snack(df, user_id, targets)

        # ------ SELECT FOODS VIA KNN (meal targets -> per-100g ideal + standardized distance) ------
        fiber_food = self.select_food_knn(df, "fiber", user_id, targets=targets)
        protein_food = self.select_food_knn(df, "protein", user_id, targets=targets)
        carb_food = self.select_food_knn(df, "carb", user_id, targets=targets)

        # side dish
        side_df = df[df["is_side"] == 1]
        side_food = None
        if len(side_df) > 0:
            side_df = side_df.copy()
            side_df["penalty"] = side_df["food_id"].apply(
                lambda fid: self.history.get_penalty(user_id, fid)
            )
            side_df = side_df.sort_values("penalty")
            side_food = side_df.iloc[0]

        foods = []
        food_labels = []

        if fiber_food is not None:
            foods.append(fiber_food)
            food_labels.append("fiber")
        if protein_food is not None:
            foods.append(protein_food)
            food_labels.append("protein")
        if carb_food is not None:
            foods.append(carb_food)
            food_labels.append("carb")

        if len(foods) == 0:
            return []

        # ------ SOLVE QUANTITIES (LP primary, sequential validation) ------
        quantities_lp = self.solve_quantities_lp(foods, targets)

        # validate and fix overflow
        quantities_lp = self.validate_and_fix(foods, quantities_lp, targets)

        # also compute sequential for comparison
        quantities_seq = self.solve_quantities_sequential(foods, targets)
        quantities_seq = self.validate_and_fix(foods, quantities_seq, targets)

        # pick the solution with lower total macro error
        def total_error(qtys):
            actual = np.zeros(4)
            for i, food in enumerate(foods):
                actual += self.macro_per_gram(food) * qtys[i]
            target_arr = np.array([
                targets["carb"], targets["protein"],
                targets["fiber"], targets["fat"]
            ])
            # weighted error
            weights = np.array([1.5, 1.5, 1.0, 1.0])
            return np.sum(weights * np.abs(actual - target_arr))

        err_lp = total_error(quantities_lp)
        err_seq = total_error(quantities_seq)

        quantities = quantities_lp if err_lp <= err_seq else quantities_seq

        # ------ BUILD RESULT ------
        result = []
        actual_macros = np.zeros(4)

        for i, food in enumerate(foods):
            qty = round(float(quantities[i]), 1)
            result.append((
                food["food_name"], qty,
                food["serving_unit"], food["food_id"]
            ))
            actual_macros += self.macro_per_gram(food) * quantities[i]

        # ------ ADD SIDE DISH IF MACRO GAPS REMAIN ------
        if side_food is not None:
            target_arr = np.array([
                targets["carb"], targets["protein"],
                targets["fiber"], targets["fat"]
            ])
            gap = target_arr - actual_macros

            if np.any(gap > 1.0):  # at least 1g gap in some macro
                side_macro = self.macro_per_gram(side_food)
                denom = np.dot(side_macro, side_macro)

                if denom > 0:
                    side_qty = np.dot(gap, side_macro) / denom
                else:
                    side_qty = 50.0

                # cap side dish so it doesn't overshoot
                for idx in range(4):
                    if side_macro[idx] > 0 and gap[idx] > 0:
                        max_side = gap[idx] / side_macro[idx]
                        side_qty = min(side_qty, max_side)

                side_qty = np.clip(side_qty, CONFIG["side_qty_min"], CONFIG["side_qty_max"])

                result.append((
                    side_food["food_name"],
                    round(float(side_qty), 1),
                    side_food["serving_unit"],
                    side_food["food_id"]
                ))

        return result

    # =====================================================
    # FEEDBACK
    # =====================================================

    def feedback(self, user_id, food_id, action):
        action = action.lower()
        if action not in ["like", "dislike", "skip"]:
            print("Invalid feedback")
            return
        self.history.add(user_id, food_id, action)
        print(f"Feedback recorded: User {user_id} Food {food_id} -> {action}")

In [53]:
def compare_macros(recommended_foods, food_df, user):

    totals = {
        "carb": 0, "protein": 0,
        "fiber": 0, "fat": 0, "calories": 0
    }

    for food_name, qty, unit, food_id in recommended_foods:
        row = food_df[food_df["food_id"] == food_id].iloc[0]
        factor = qty / 100.0
        totals["carb"] += row["carb_g"] * factor
        totals["protein"] += row["protein_g"] * factor
        totals["fiber"] += row["fiber_g"] * factor
        totals["fat"] += row["fat_g"] * factor
        totals["calories"] += row["energy_kcal"] * factor

    print("\n" + "=" * 50)
    print("MACRO COMPARISON")
    print("=" * 50)

    def show(name, target, actual):
        pct = (actual / target) * 100 if target > 0 else 0
        status = "✅" if 85 <= pct <= 115 else "⚠️"
        print(f"\n{status} {name}")
        print(f"   Target  : {target:.1f} g")
        print(f"   Actual  : {actual:.1f} g")
        print(f"   Achieved: {pct:.1f} %")

    show("Carbs", user["target_carbs_g"], totals["carb"])
    show("Protein", user["target_protein_g"], totals["protein"])
    show("Fiber", user["target_fiber_g"], totals["fiber"])
    show("Fat", user["target_fat_g"], totals["fat"])
    show("Calories", user["target_calories_kcal"], totals["calories"])

    print("\n" + "=" * 50)

In [54]:
if __name__ == "__main__":

    df = pd.read_csv("./south_indian_food_with_id.csv")

    recommender = FoodRecommender(df)

    # ================== USER DATABASE ==================

    users = {

        1: {
            "name": "Dhanush",
            "veg_only": True,
            "hba1c_percent": 7.2,
            "triglycerides_mg_dl": 180,
            "ldl_cholesterol_mg_dl": 140,
            "systolic_bp_mmHg": 150,
            "diastolic_bp_mmHg": 95,
            "targets": {
                "breakfast": {"carbs": 40, "protein": 20, "fat": 10, "fiber": 12, "calories": 350},
                "lunch":     {"carbs": 60, "protein": 25, "fat": 15, "fiber": 15, "calories": 500},
                "snacks":    {"carbs": 25, "protein": 10, "fat": 8,  "fiber": 8,  "calories": 200},
                "dinner":    {"carbs": 45, "protein": 22, "fat": 12, "fiber": 12, "calories": 400}
            }
        },

        2: {
            "name": "Ravi",
            "veg_only": False,
            "hba1c_percent": 6.4,
            "triglycerides_mg_dl": 130,
            "ldl_cholesterol_mg_dl": 110,
            "systolic_bp_mmHg": 130,
            "diastolic_bp_mmHg": 85,
            "targets": {
                "breakfast": {"carbs": 50, "protein": 18, "fat": 12, "fiber": 10, "calories": 400},
                "lunch":     {"carbs": 70, "protein": 30, "fat": 18, "fiber": 16, "calories": 600},
                "snacks":    {"carbs": 30, "protein": 12, "fat": 10, "fiber": 8,  "calories": 250},
                "dinner":    {"carbs": 50, "protein": 25, "fat": 14, "fiber": 12, "calories": 450}
            }
        }
    }

    # ================== USER LOGIN ==================

    print("\nAvailable Users:")
    for uid in users:
        print(f"  {uid} → {users[uid]['name']}")

    user_id = int(input("\nEnter User ID: "))

    if user_id not in users:
        print("Invalid User")
        exit()

    user_data = users[user_id]
    print(f"\nWelcome {user_data['name']}!")

    # ================== MEAL SELECTION ==================

    print("\nSelect Meal Type:")
    print("  1 → Breakfast")
    print("  2 → Lunch")
    print("  3 → Snacks")
    print("  4 → Dinner")

    meal_map = {"1": "breakfast", "2": "lunch", "3": "snacks", "4": "dinner"}
    meal_choice = input("Enter choice: ").strip()
    meal_type = meal_map.get(meal_choice, "breakfast")
    meal_targets = user_data["targets"][meal_type]

    # ================== BUILD USER INPUT ==================

    user = {
        "meal_type": meal_type,
        "veg_only": user_data["veg_only"],
        "target_carbs_g": meal_targets["carbs"],
        "target_protein_g": meal_targets["protein"],
        "target_fat_g": meal_targets["fat"],
        "target_fiber_g": meal_targets["fiber"],
        "target_calories_kcal": meal_targets["calories"],
        "hba1c_percent": user_data["hba1c_percent"],
        "triglycerides_mg_dl": user_data["triglycerides_mg_dl"],
        "ldl_cholesterol_mg_dl": user_data["ldl_cholesterol_mg_dl"],
        "systolic_bp_mmHg": user_data["systolic_bp_mmHg"],
        "diastolic_bp_mmHg": user_data["diastolic_bp_mmHg"]
    }

    # ================== RECOMMENDATION ==================

    foods = recommender.recommend(user_id, user)

    print(f"\n{'=' * 50}")
    print(f"Recommended {meal_type.title()}")
    print(f"{'=' * 50}")

    for f in foods:
        print(f"  {f[0]} → {f[1]} g ({f[2]})")

    # ================== MACRO COMPARISON ==================

    compare_macros(foods, df, user)

    # ================== FEEDBACK ==================

    print("\n--- Feedback ---")

    for f in foods:
        food_name, qty, unit, food_id = f
        print(f"\nFood: {food_name}")
        print("  1 → Like  |  2 → Dislike  |  3 → Skip")
        choice = input("  Select (1/2/3): ").strip()
        action_map = {"1": "like", "2": "dislike", "3": "skip"}
        action = action_map.get(choice, "skip")
        recommender.feedback(user_id, food_id, action)

    print("\n--- Updated History ---")
    print(recommender.history.history)
    


Available Users:
  1 → Dhanush
  2 → Ravi

Welcome Dhanush!

Select Meal Type:
  1 → Breakfast
  2 → Lunch
  3 → Snacks
  4 → Dinner

Recommended Dinner
  Vendhaya Keerai (Fenugreek Leaves) → 250.0 g (bowl)
  Sprouted Horse Gram Sundal → 30.0 g (bowl)
  Adai Dosai → 92.5 g (piece)
  Coriander Chutney → 50.0 g (tablespoon)

MACRO COMPARISON

✅ Carbs
   Target  : 45.0 g
   Actual  : 48.4 g
   Achieved: 107.6 %

✅ Protein
   Target  : 22.0 g
   Actual  : 22.8 g
   Achieved: 103.8 %

⚠️ Fiber
   Target  : 12.0 g
   Actual  : 14.3 g
   Achieved: 119.2 %

⚠️ Fat
   Target  : 12.0 g
   Actual  : 6.8 g
   Achieved: 57.0 %

✅ Calories
   Target  : 400.0 g
   Actual  : 342.5 g
   Achieved: 85.6 %


--- Feedback ---

Food: Vendhaya Keerai (Fenugreek Leaves)
  1 → Like  |  2 → Dislike  |  3 → Skip
Feedback recorded: User 1 Food 32 -> like

Food: Sprouted Horse Gram Sundal
  1 → Like  |  2 → Dislike  |  3 → Skip
Feedback recorded: User 1 Food 86 -> like

Food: Adai Dosai
  1 → Like  |  2 → Dislike